# 04 — Publication-Quality Figures with Full Benchmark Names

This notebook **only reads results that already exist** in
`improved_experiment_v2/results/` (written by notebooks 02 and 03). Nothing is
retrained and no ray tracing is repeated. The per-fold policy metrics are
recomputed from the saved fold predictions (a few seconds) using exactly the
same policy code and settings as notebook 03, so the numbers match the paper.

Compared with the earlier all-bar-chart figures, the plots here

* use the **full benchmark names** (Persistence, Deterministic GRU, Independent
  Probabilistic GRU, Joint Probabilistic GRU, Hurdle-Aware Joint GRU) and full
  policy names instead of code identifiers such as `hurdle_joint`;
* use a plot type suited to each quantity: a **strip and mean plot** showing every
  fold for forecast error, **dumbbell plots** for the legacy versus proposed
  policy comparison, a **trade-off scatter** with arrows, and a **per-fold heatmap**;
* share one aligned label column, drop long in-figure titles (captions carry
  them), and use serif fonts, colour-blind-safe colours, and vector PDF output
  sized for IEEE double-column layout.

Outputs go to `improved_experiment_v2/figures_pro/`.


In [ ]:
from pathlib import Path
import csv
import json
import textwrap

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

PROJECT_ROOT = Path.home() / "wcnc_project"
V2_RESULTS_DIR = PROJECT_ROOT / "improved_experiment_v2" / "results"
FIG_DIR = PROJECT_ROOT / "improved_experiment_v2" / "figures_pro"
FIG_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.alpha": 0.25, "grid.linewidth": 0.4,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight", "savefig.dpi": 300,
})

# ---------- full names ----------
MODEL_ORDER = ["persistence", "deterministic", "independent", "joint", "hurdle_joint"]
FULL_NAME = {
    "persistence": "Persistence",
    "deterministic": "Deterministic GRU",
    "independent": "Independent Probabilistic GRU",
    "joint": "Joint Probabilistic GRU",
    "hurdle_joint": "Hurdle-Aware Joint GRU",
}
POLICY_ORDER = ["legacy_margin", "dp_ttt"]
POLICY_NAME = {
    "legacy_margin": "Legacy average-score margin policy",
    "dp_ttt": "Proposed DP-TTT policy",
}
# colour-blind-safe (Okabe-Ito): grey baselines, vermilion = non-hurdle joint, blue = proposed
COLOR = {
    "persistence": "#7F7F7F", "deterministic": "#009E73", "independent": "#E69F00",
    "joint": "#D55E00", "hurdle_joint": "#0072B2",
}

def wrap(name, width=17):
    return textwrap.fill(name, width=width)

Y_LABELS = [wrap(FULL_NAME[m]) for m in MODEL_ORDER]
print("Figures will be written to:", FIG_DIR)


In [ ]:
# ------------------------------------------------------------
# Load per-fold forecast metrics saved by notebook 02
# ------------------------------------------------------------
with open(V2_RESULTS_DIR / "forecaster_loto_summary.json") as f:
    forecast_summary = json.load(f)

NUM_FOLDS = forecast_summary["num_folds"]
mae_folds = {
    m: np.array([forecast_summary["per_fold_results"][str(k)][m]["masked_mae_db"]
                 for k in range(NUM_FOLDS)], dtype=float)
    for m in MODEL_ORDER
}
print("Loaded masked MAE for", NUM_FOLDS, "folds")


In [ ]:
# ------------------------------------------------------------
# Recompute per-fold policy metrics from saved predictions.
# Policy code and settings are identical to notebook 03.
# Two extra valid-link metrics are added (G_valid and I_invalid).
# ------------------------------------------------------------
RISK_AVERSION = 0.5
HANDOVER_MARGIN_DB = 2.0
HANDOVER_COST_DB = 2.0
TTT_K = 2
LINK_GATE_THRESHOLD = None


def build_score_matrices(mean_db, std_db=None, link_prob=None, risk_aversion=0.5, link_gate_threshold=None):
    score = mean_db.copy()
    if std_db is not None:
        score = score - risk_aversion * std_db
    if link_prob is not None:
        score = score + 10.0 * np.log1p(link_prob) - 10.0 * np.log1p(np.ones_like(link_prob))
        if link_gate_threshold is not None:
            score = np.where(link_prob < link_gate_threshold, -1e6, score)
    return score


def run_legacy_margin_policy(score_avg, current_power_db, margin_db=2.0):
    raw_actions = np.argmax(score_avg, axis=1).astype(np.int16)
    actions = np.zeros(len(score_avg), dtype=np.int16)
    serving = int(np.argmax(current_power_db[0]))
    for i in range(len(score_avg)):
        cand = int(raw_actions[i])
        if cand != serving and score_avg[i, cand] - score_avg[i, serving] >= margin_db:
            serving = cand
        actions[i] = serving
    return actions


def _dp_first_action(step_scores, current_serving, handover_cost):
    horizon, num_gnbs = step_scores.shape
    value_next = np.zeros(num_gnbs)
    first_policy = None
    for h in range(horizon - 1, -1, -1):
        value_cur = np.zeros(num_gnbs)
        action_cur = np.zeros(num_gnbs, dtype=np.int16)
        for g_prev in range(num_gnbs):
            best_v, best_g = -np.inf, g_prev
            for g_cur in range(num_gnbs):
                cost = handover_cost if g_cur != g_prev else 0.0
                v = step_scores[h, g_cur] - cost + value_next[g_cur]
                if v > best_v:
                    best_v, best_g = v, g_cur
            value_cur[g_prev] = best_v
            action_cur[g_prev] = best_g
        if h == 0:
            first_policy = action_cur
        value_next = value_cur
    return first_policy[current_serving]


def run_dp_ttt_policy(step_scores, current_power_db, handover_cost_db=2.0, ttt_k=2):
    n = step_scores.shape[0]
    actions = np.zeros(n, dtype=np.int16)
    serving = int(np.argmax(current_power_db[0]))
    pending, streak = None, 0
    for i in range(n):
        rec = _dp_first_action(step_scores[i], serving, handover_cost_db)
        if rec == serving:
            pending, streak = None, 0
        else:
            if rec == pending:
                streak += 1
            else:
                pending, streak = rec, 1
            if streak >= ttt_k:
                serving, pending, streak = rec, None, 0
        actions[i] = serving
    return actions


def evaluate_policy(actions, target_power_db, test_valid, future_best_gnb):
    n, horizon, _ = target_power_db.shape
    oi, si = np.arange(n), np.arange(horizon)
    sel_power = target_power_db[oi[:, None], si[None, :], actions[:, None]]
    sel_valid = test_valid[oi[:, None], si[None, :], actions[:, None]] > 0
    oracle_gap = np.max(target_power_db, axis=2) - sel_power

    # valid-link metrics: gap to the best *usable* link, and how often an unusable link is chosen
    usable_power = np.where(test_valid > 0, target_power_db, -np.inf)
    best_usable = usable_power.max(axis=2)
    ok = np.isfinite(best_usable) & sel_valid
    g_valid = float((best_usable - sel_power)[ok].mean()) if ok.any() else float("nan")

    return {
        "mean_oracle_gap_db": float(oracle_gap.mean()),
        "handover_count": int(np.sum(np.diff(actions) != 0)),
        "valid_link_gap_db": g_valid,
        "invalid_action_rate": float(1.0 - sel_valid.mean()),
    }


per_fold_policy = {m: {p: [] for p in POLICY_ORDER} for m in MODEL_ORDER}
for k in range(NUM_FOLDS):
    with np.load(V2_RESULTS_DIR / f"fold_{k}_summary_predictions.npz", allow_pickle=True) as d:
        files = list(d.files)
        target_db, test_valid, test_best = d["test_target_db"], d["test_Yv"], d["test_best"]
        for m in MODEL_ORDER:
            mean_db = d[f"{m}_mean_db"]
            std_db = d[f"{m}_std_db"] if f"{m}_std_db" in files else None
            link_prob = d[f"{m}_link_prob"] if f"{m}_link_prob" in files else None
            cur = mean_db[:, 0, :]
            score_step = build_score_matrices(mean_db, std_db, link_prob, RISK_AVERSION, LINK_GATE_THRESHOLD)
            a_leg = run_legacy_margin_policy(score_step.mean(axis=1), cur, HANDOVER_MARGIN_DB)
            a_dp = run_dp_ttt_policy(score_step, cur, HANDOVER_COST_DB, TTT_K)
            per_fold_policy[m]["legacy_margin"].append(evaluate_policy(a_leg, target_db, test_valid, test_best))
            per_fold_policy[m]["dp_ttt"].append(evaluate_policy(a_dp, target_db, test_valid, test_best))


def arr(model, policy, metric):
    return np.array([r[metric] for r in per_fold_policy[model][policy]], dtype=float)


def mean_std(x):
    x = x[~np.isnan(x)]
    return float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else 0.0


with open(V2_RESULTS_DIR / "policy_perfold_metrics.json", "w", encoding="utf-8") as f:
    json.dump(per_fold_policy, f, indent=2)
print("Recomputed per-fold policy metrics for", NUM_FOLDS, "folds x", len(MODEL_ORDER), "forecasters x 2 policies")


## Figure A — main result (three panels, one shared label column)

* **(a)** masked forecast MAE: every fold is a small dot, the diamond is the fold
  mean, whiskers are one standard deviation, and the stem from zero makes the
  ranking easy to read. A paired t-test of the proposed model against the
  non-hurdle joint model is printed inside the panel.
* **(b, c)** handover count and oracle gap as **dumbbells**: the hollow marker is the
  legacy margin policy, the filled marker is the proposed DP-TTT policy, and the
  connecting line shows the effect of the policy for that forecaster.


In [ ]:
ypos = np.arange(len(MODEL_ORDER))[::-1].astype(float)
rng = np.random.default_rng(7)

fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.55), sharey=True,
                         gridspec_kw={"wspace": 0.10, "width_ratios": [1.0, 1.0, 1.0]})

# ---- (a) masked MAE: strip + mean +/- std lollipop ----
ax = axes[0]
for m, y in zip(MODEL_ORDER, ypos):
    vals = mae_folds[m]
    mu, sd = mean_std(vals)
    ax.hlines(y, 0, mu, color=COLOR[m], lw=1.2, alpha=0.55, zorder=1)
    ax.scatter(vals, y + rng.uniform(-0.13, 0.13, len(vals)), s=9, color=COLOR[m],
               alpha=0.45, linewidths=0, zorder=2)
    ax.errorbar(mu, y, xerr=sd, fmt="D", ms=5, color=COLOR[m], mec="black", mew=0.5,
                ecolor="black", elinewidth=0.8, capsize=2, zorder=3)
    ax.text(mu + sd + 0.35, y, f"{mu:.1f}", va="center", fontsize=6.5)
mu_h, _ = mean_std(mae_folds["hurdle_joint"])
mu_j, _ = mean_std(mae_folds["joint"])
t_stat, p_val = stats.ttest_rel(mae_folds["joint"], mae_folds["hurdle_joint"])
ax.text(0.98, 0.04,
        f"vs. Joint GRU: {100 * (1 - mu_h / mu_j):.0f}% lower\npaired $p$ = {p_val:.1e}",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=6.3,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", lw=0.5))
ax.set_xlim(0, max(mean_std(v)[0] + mean_std(v)[1] for v in mae_folds.values()) * 1.25)
ax.set_xlabel("Masked forecast MAE (dB)")
ax.set_title("(a)", loc="left", fontweight="bold")
ax.set_yticks(ypos)
ax.set_yticklabels(Y_LABELS)
ax.get_yticklabels()[-1].set_fontweight("bold")
ax.tick_params(axis="y", length=0)


def dumbbell(ax, metric, xlabel, title, decimals):
    for m, y in zip(MODEL_ORDER, ypos):
        mu_l, sd_l = mean_std(arr(m, "legacy_margin", metric))
        mu_d, sd_d = mean_std(arr(m, "dp_ttt", metric))
        ax.plot([mu_l, mu_d], [y, y], color=COLOR[m], lw=2.0, alpha=0.6, zorder=1,
                solid_capstyle="round")
        ax.errorbar(mu_l, y, xerr=sd_l, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.errorbar(mu_d, y, xerr=sd_d, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.scatter(mu_l, y, s=34, facecolor="white", edgecolor=COLOR[m], linewidths=1.3, zorder=3)
        ax.scatter(mu_d, y, s=34, facecolor=COLOR[m], edgecolor="black", linewidths=0.5, zorder=4)
    ax.set_xlim(left=0)
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.tick_params(axis="y", length=0)


dumbbell(axes[1], "handover_count", "Target-cell switches per trajectory", "(b)", 1)
dumbbell(axes[2], "mean_oracle_gap_db", "Mean oracle gap (dB)", "(c)", 1)

policy_handles = [
    Line2D([0], [0], marker="o", ls="", mfc="white", mec="black", mew=1.2, ms=5.5, label=POLICY_NAME["legacy_margin"]),
    Line2D([0], [0], marker="o", ls="", mfc="black", mec="black", ms=5.5, label=POLICY_NAME["dp_ttt"]),
]
fig.legend(handles=policy_handles, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.56, -0.07), handletextpad=0.3, columnspacing=1.8)
fig.subplots_adjust(left=0.20, right=0.99, top=0.93, bottom=0.20)

fig.savefig(FIG_DIR / "fig_main_loto_diagnostics.pdf")
fig.savefig(FIG_DIR / "fig_main_loto_diagnostics.png")
plt.show()


## Figure B — policy trade-off scatter

Each forecaster appears twice, joined by an arrow from the legacy policy (hollow)
to the proposed DP-TTT policy (filled). Lower-left is better on both axes, so the
plot shows in one view whether a policy change trades handover count against
oracle gap. Crosshairs are one standard deviation across folds.


In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.9))
ax.grid(True, axis="both")
for m in MODEL_ORDER:
    x_l, xs_l = mean_std(arr(m, "legacy_margin", "handover_count"))
    y_l, ys_l = mean_std(arr(m, "legacy_margin", "mean_oracle_gap_db"))
    x_d, xs_d = mean_std(arr(m, "dp_ttt", "handover_count"))
    y_d, ys_d = mean_std(arr(m, "dp_ttt", "mean_oracle_gap_db"))
    for (x, y, xs, ys) in [(x_l, y_l, xs_l, ys_l), (x_d, y_d, xs_d, ys_d)]:
        ax.errorbar(x, y, xerr=xs, yerr=ys, fmt="none", ecolor=COLOR[m], elinewidth=0.6, alpha=0.5, capsize=0)
    ax.annotate("", xy=(x_d, y_d), xytext=(x_l, y_l),
                arrowprops=dict(arrowstyle="-|>", color=COLOR[m], lw=1.0, shrinkA=3, shrinkB=3, alpha=0.9))
    ax.scatter(x_l, y_l, s=38, facecolor="white", edgecolor=COLOR[m], linewidths=1.4, zorder=3)
    ax.scatter(x_d, y_d, s=38, facecolor=COLOR[m], edgecolor="black", linewidths=0.5, zorder=4)

ax.set_xlabel("Target-cell switches per trajectory")
ax.set_ylabel("Mean oracle gap (dB)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.text(0.03, 0.03, "better", transform=ax.transAxes, fontsize=7, style="italic", color="0.35")

model_handles = [Line2D([0], [0], marker="s", ls="", color=COLOR[m], ms=5, label=FULL_NAME[m]) for m in MODEL_ORDER]
policy_handles = [
    Line2D([0], [0], marker="o", ls="", mfc="white", mec="black", mew=1.2, ms=5, label="Legacy policy"),
    Line2D([0], [0], marker="o", ls="", mfc="black", mec="black", ms=5, label="Proposed DP-TTT policy"),
]
ax.legend(handles=model_handles + policy_handles, loc="upper center", bbox_to_anchor=(0.5, -0.22),
          ncol=2, frameon=False, fontsize=6.3, columnspacing=1.2, handletextpad=0.3)
fig.savefig(FIG_DIR / "fig_policy_tradeoff_scatter.pdf")
fig.savefig(FIG_DIR / "fig_policy_tradeoff_scatter.png")
plt.show()


## Figure C — per-fold masked MAE heatmap

Shows every held-out trajectory for every forecaster, so a reader can see that the
ranking is consistent across folds and not driven by a single test trajectory.


In [ ]:
data = np.array([mae_folds[m] for m in MODEL_ORDER])          # models x folds
means = data.mean(axis=1, keepdims=True)
full = np.hstack([data, means])

fig, ax = plt.subplots(figsize=(3.5, 2.2))
ax.grid(False)
im = ax.imshow(full, cmap="viridis_r", aspect="auto")
ax.axvline(NUM_FOLDS - 0.5, color="white", lw=2.0)
for i in range(full.shape[0]):
    for j in range(full.shape[1]):
        v = full[i, j]
        r, g, b, _ = im.cmap(im.norm(v))
        col = "black" if (0.299 * r + 0.587 * g + 0.114 * b) > 0.55 else "white"
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=6.5, color=col,
                fontweight="bold" if j == NUM_FOLDS else "normal")
ax.set_xticks(range(NUM_FOLDS + 1))
ax.set_xticklabels([f"F{k}" for k in range(NUM_FOLDS)] + ["Mean"])
ax.set_yticks(range(len(MODEL_ORDER)))
ax.set_yticklabels([wrap(FULL_NAME[m], 20) for m in MODEL_ORDER])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xlabel("Held-out trajectory (fold)")
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cb.set_label("Masked MAE (dB)", fontsize=7)
cb.ax.tick_params(labelsize=6)
fig.savefig(FIG_DIR / "fig_masked_mae_fold_heatmap.pdf")
fig.savefig(FIG_DIR / "fig_masked_mae_fold_heatmap.png")
plt.show()


## Figure D — valid-link policy metrics (supplementary)

The paper notes that the sentinel-inclusive oracle gap can mislead when a policy
selects a link with no usable path. This figure separates the two effects:
**(a)** the gap to the best *usable* link when the chosen link is usable, and
**(b)** how often the policy chooses an unusable link.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5.4, 2.4), sharey=True, gridspec_kw={"wspace": 0.10})
ypos = np.arange(len(MODEL_ORDER))[::-1].astype(float)


def dumbbell2(ax, metric, xlabel, title, scale=1.0):
    for m, y in zip(MODEL_ORDER, ypos):
        mu_l, sd_l = mean_std(arr(m, "legacy_margin", metric))
        mu_d, sd_d = mean_std(arr(m, "dp_ttt", metric))
        mu_l, sd_l, mu_d, sd_d = mu_l * scale, sd_l * scale, mu_d * scale, sd_d * scale
        ax.plot([mu_l, mu_d], [y, y], color=COLOR[m], lw=2.0, alpha=0.6, zorder=1, solid_capstyle="round")
        ax.errorbar(mu_l, y, xerr=sd_l, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.errorbar(mu_d, y, xerr=sd_d, fmt="none", ecolor=COLOR[m], elinewidth=0.7, capsize=1.6, alpha=0.8)
        ax.scatter(mu_l, y, s=34, facecolor="white", edgecolor=COLOR[m], linewidths=1.3, zorder=3)
        ax.scatter(mu_d, y, s=34, facecolor=COLOR[m], edgecolor="black", linewidths=0.5, zorder=4)
    ax.set_xlim(left=0)
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.tick_params(axis="y", length=0)


dumbbell2(axes[0], "valid_link_gap_db", "Gap to best usable link (dB)", "(a)")
dumbbell2(axes[1], "invalid_action_rate", "Unusable-link selections (%)", "(b)", scale=100.0)
axes[0].set_yticks(ypos)
axes[0].set_yticklabels(Y_LABELS)
axes[0].get_yticklabels()[-1].set_fontweight("bold")
fig.legend(handles=policy_handles, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.58, -0.08), handletextpad=0.3, columnspacing=1.8)
fig.subplots_adjust(left=0.30, right=0.99, top=0.92, bottom=0.22)
fig.savefig(FIG_DIR / "fig_valid_link_policy_metrics.pdf")
fig.savefig(FIG_DIR / "fig_valid_link_policy_metrics.png")
plt.show()


In [ ]:
# ------------------------------------------------------------
# Summary table with full names (CSV) for the paper
# ------------------------------------------------------------
rows = []
for m in MODEL_ORDER:
    mu, sd = mean_std(mae_folds[m])
    for p in POLICY_ORDER:
        rows.append({
            "Forecaster": FULL_NAME[m], "Policy": POLICY_NAME[p],
            "Masked MAE (dB)": f"{mu:.2f} +/- {sd:.2f}",
            "Switches per trajectory": "%.1f +/- %.1f" % mean_std(arr(m, p, "handover_count")),
            "Oracle gap (dB)": "%.2f +/- %.2f" % mean_std(arr(m, p, "mean_oracle_gap_db")),
            "Gap to best usable link (dB)": "%.2f +/- %.2f" % mean_std(arr(m, p, "valid_link_gap_db")),
            "Unusable-link selections (%)": "%.1f +/- %.1f" % tuple(np.array(mean_std(arr(m, p, "invalid_action_rate"))) * 100),
        })

with open(FIG_DIR / "results_summary_full_names.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

for r in rows:
    print(f"{r['Forecaster']:32s} | {r['Policy']:36s} | MAE {r['Masked MAE (dB)']:>14s} | "
          f"switches {r['Switches per trajectory']:>12s} | gap {r['Oracle gap (dB)']:>13s}")
print("\nSaved figures and table to:", FIG_DIR)
